In [ ]:
import pandas as pd
inst = pd.read_csv("instrumento_ola2_meindl_v2.csv")
print("Instrumento cargado:", len(inst), "ítems")
print("Bloques:", inst.groupby(['bloque','formato']).size().to_dict())

SYS = {}
SYS["escala"] = """You are completing a structured research questionnaire about distributive justice. For each statement or scenario, respond with a single whole number on the scale indicated (1 to 7). Read each item to see what the endpoints mean.

Format your response exactly as:
SCORE: <number>
REASON: <one short sentence>

Output nothing before "SCORE:" and nothing after the reason sentence."""

SYS["binario"] = """You are completing a structured research questionnaire about distributive justice. You will read a scenario with two proposals, A and B. Choose which one you find fairer.

Format your response exactly as:
CHOICE: <A or B>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

SYS["menu"] = """You are completing a structured research questionnaire about distributing resources. You will be shown several distribution rules. Choose the single rule you most prefer.

Format your response exactly as:
CHOICE: <number>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

print("System prompts listos:", list(SYS.keys()))

Instrumento cargado: 65 ítems
Bloques: {('A_declarado_cdjs_largo', 'escala'): 36, ('B_dilema', 'binario'): 7, ('B_dilema', 'escala'): 21, ('C_regla_conductual', 'menu'): 1}
System prompts listos: ['escala', 'binario', 'menu']


In [ ]:
import re

def parsear(texto, formato):
    """Devuelve (valor, estado). valor es int (escala/menu) o 'A'/'B' (binario)."""
    if texto is None:
        return None, "error"
    t = texto.strip()
    if any(n in t.lower() for n in ["i can't","i cannot","i'm unable","i won't"]) and "SCORE:" not in t and "CHOICE:" not in t:
        return None, "negativa"

    if formato == "escala":
        m = re.findall(r"SCORE:\s*([1-7])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-7])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

    if formato == "binario":
        m = re.findall(r"CHOICE:\s*([AB])\b", t)
        if len(m) == 1: return m[0], "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([AB])\b", t)
        return (m2[0], "ok") if m2 else (None, "sin_letra")

    if formato == "menu":
        m = re.findall(r"CHOICE:\s*([1-4])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-4])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

    return None, "formato_desconocido"

print(parsear("SCORE: 5\nREASON: x", "escala"))
print(parsear("CHOICE: A\nREASON: x", "binario"))
print(parsear("CHOICE: 3\nREASON: x", "menu"))

(5, 'ok')
('A', 'ok')
(3, 'ok')


In [ ]:
def llamar_glm_ola2(item_texto, system_prompt):
    """GLM 5.2 modo máximo (xhigh) para la Ola 2. Devuelve (texto, error)."""
    try:
        resp = cliente_glm.chat.completions.create(
            model="glm-5.2",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item_texto},
            ],
            max_tokens=800,
            extra_body={"reasoning_effort": "xhigh"}
        )
        return resp.choices[0].message.content, None
    except Exception as e:
        return None, str(e)

print("Función llamar_glm_ola2 lista.")

Función llamar_glm_ola2 lista.


In [ ]:
# Tomamos un ítem de cada formato del instrumento
ejemplos = {}
for fmt in ["escala", "binario", "menu"]:
    fila = inst[inst.formato == fmt].iloc[0]
    ejemplos[fmt] = fila

print("Prueba de humo GLM — 3 formatos:\n")
for fmt, item in ejemplos.items():
    sysprompt = SYS[fmt]
    out, err = llamar_glm_ola2(item["texto"], sysprompt)
    if err:
        print(f"  {fmt:8s} ({item['item_id']}): ❌ ERROR {err[:80]}")
    else:
        valor, estado = parsear(out, fmt)
        print(f"  {fmt:8s} ({item['item_id']}): valor={valor} [{estado}]")
        print(f"           crudo: {repr(out[:70])}")

Prueba de humo GLM — 3 formatos:

  escala   (CDJS_Equali_T): ❌ ERROR name 'cliente_glm' is not defined
  binario  (DIL_01_binario): ❌ ERROR name 'cliente_glm' is not defined
  menu     (REGLA_menu): ❌ ERROR name 'cliente_glm' is not defined


In [ ]:
!pip install --upgrade openai --quiet
from google.colab import userdata
from openai import OpenAI

cliente_glm = OpenAI(
    api_key=userdata.get("ZAI_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/"
)
print("Cliente GLM listo.")

Cliente GLM listo.


In [ ]:
faltan = []
for nombre in ["cliente_glm", "inst", "SYS", "parsear", "llamar_glm_ola2"]:
    if nombre not in dir():
        faltan.append(nombre)
print("✅ Todo cargado" if not faltan else f"⚠️ Faltan por reejecutar: {faltan}")

✅ Todo cargado


In [ ]:
# Tomamos un ítem de cada formato del instrumento
ejemplos = {}
for fmt in ["escala", "binario", "menu"]:
    fila = inst[inst.formato == fmt].iloc[0]
    ejemplos[fmt] = fila

print("Prueba de humo GLM — 3 formatos:\n")
for fmt, item in ejemplos.items():
    sysprompt = SYS[fmt]
    out, err = llamar_glm_ola2(item["texto"], sysprompt)
    if err:
        print(f"  {fmt:8s} ({item['item_id']}): ❌ ERROR {err[:80]}")
    else:
        valor, estado = parsear(out, fmt)
        print(f"  {fmt:8s} ({item['item_id']}): valor={valor} [{estado}]")
        print(f"           crudo: {repr(out[:70])}")

Prueba de humo GLM — 3 formatos:

  escala   (CDJS_Equali_T): valor=3 [ok]
           crudo: 'SCORE: 3\nREASON: Treating everyone identically regardless of effort, c'
  binario  (DIL_01_binario): valor=A [ok]
           crudo: 'CHOICE: A\nREASON: Tying bonuses to measurable contributions rewards em'
  menu     (REGLA_menu): valor=1 [ok]
           crudo: 'CHOICE: 1\nREASON: Paying based on quality of work rewards excellence a'


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

filas, hecho = [], 0
total = len(inst) * 10
print(f"Voy a hacer {total} llamadas (GLM, Ola 2, instrumento principal).\n")

for _, item in inst.iterrows():
    fmt = item["formato"]
    sysprompt = SYS[fmt]
    for rep in range(10):
        out, err = llamar_glm_ola2(item["texto"], sysprompt)
        valor, estado = parsear(out, fmt)
        filas.append({
            "modelo": "glm-5.2", "modo": "xhigh",
            "item_id": item["item_id"], "bloque": item["bloque"],
            "formato": fmt, "principio": item["principio"], "limpieza": item["limpieza"],
            "repeticion": rep, "valor": valor,
            "estado_parseo": estado if err is None else "error",
            "error": err, "respuesta_cruda": out,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        hecho += 1
        if hecho % 50 == 0:
            print(f"  progreso: {hecho}/{total}")
            pd.DataFrame(filas).to_csv("glm_ola2_crudo.csv", index=False)
        time.sleep(0.3)

df = pd.DataFrame(filas)
df.to_csv("glm_ola2_crudo.csv", index=False)
print(f"\nListo. {len(df)} filas en glm_ola2_crudo.csv")
print("\nEstados de parseo por formato:")
print(df.groupby(["formato","estado_parseo"]).size())

Voy a hacer 650 llamadas (GLM, Ola 2, instrumento principal).

  progreso: 50/650
  progreso: 100/650
  progreso: 150/650
  progreso: 200/650
  progreso: 250/650
  progreso: 300/650
  progreso: 350/650
  progreso: 400/650
  progreso: 450/650
  progreso: 500/650
  progreso: 550/650
  progreso: 600/650
  progreso: 650/650

Listo. 650 filas en glm_ola2_crudo.csv

Estados de parseo por formato:
formato  estado_parseo
binario  ok                70
escala   ok               561
         sin_numero         9
menu     ok                10
dtype: int64


In [ ]:
from openai import OpenAI
cliente_glm = OpenAI(
    api_key=userdata.get("ZAI_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/"
)

NameError: name 'userdata' is not defined

In [ ]:
from google.colab import userdata
from openai import OpenAI

cliente_glm = OpenAI(
    api_key=userdata.get("ZAI_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/"
)
print("Cliente GLM listo.")

Cliente GLM listo.


In [ ]:
faltan = [n for n in ["cliente_glm","sys_menu3","parsear_menu3","items_3op","clave"] if n not in dir()]
print("✅ Todo cargado" if not faltan else f"⚠️ Faltan por reejecutar: {faltan}")

⚠️ Faltan por reejecutar: ['sys_menu3', 'parsear_menu3', 'items_3op', 'clave']


In [ ]:
import re

sys_menu3 = """You are completing a structured research questionnaire about distributing resources. You will be shown a scenario with three possible distribution rules. Choose the single rule you find fairest.

Format your response exactly as:
CHOICE: <1, 2, or 3>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

def parsear_menu3(texto):
    if texto is None:
        return None, "error"
    t = texto.strip()
    m = re.findall(r"CHOICE:\s*([1-3])\b", t)
    if len(m) == 1: return int(m[0]), "ok"
    if len(m) > 1: return None, "ambiguo"
    m2 = re.findall(r"\b([1-3])\b", t)
    return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

items_3op = {
    "DIL3OP_01_bonos": ("The Altaia company sets aside a portion of profits as a yearly bonus for its employees. "
        "The board wants to choose the fairest way to divide it (the choice doesn't change the total pool). "
        "Which distribution rule is fairest?\n"
        "1) Based on contribution — employees who contribute more to the company's success receive a larger share.\n"
        "2) Equal — every employee receives the same share.\n"
        "3) Based on need — employees in greater financial need receive a larger share."),
    "DIL3OP_03_fondos": ("The John Henry Dean Foundation distributes a large grant among several charities each year. "
        "The board wants the fairest criterion (the choice doesn't change the total funds). "
        "Which distribution rule is fairest?\n"
        "1) Equal — every charity receives the same amount.\n"
        "2) Based on need — charities whose beneficiaries are in the most desperate circumstances receive more.\n"
        "3) Based on results — charities that produce better results with the money receive more."),
    "DIL3OP_07_sede": ("The International Athletics Council chooses which member country hosts its yearly event, which "
        "gives an economic boost to the host. They want the fairest criterion (the choice doesn't change "
        "any country's dues). Which rule for choosing the host is fairest?\n"
        "1) Based on need — the country whose economy most needs the boost is chosen.\n"
        "2) Based on capability — the country with the best facilities to ensure the event's success is chosen.\n"
        "3) Equal — countries rotate so each gets an equal opportunity to host."),
}

clave = {
    "DIL3OP_01_bonos":  {1:"merito", 2:"igualdad", 3:"necesidad"},
    "DIL3OP_03_fondos": {1:"igualdad", 2:"necesidad", 3:"merito"},
    "DIL3OP_07_sede":   {1:"necesidad", 2:"merito", 3:"igualdad"},
}
print("Escenarios, prompt y parser de 3 opciones listos.")
print(parsear_menu3("CHOICE: 2\nREASON: x"))

Escenarios, prompt y parser de 3 opciones listos.
(2, 'ok')


In [ ]:
print("Prueba de humo GLM — 3 escenarios de opción múltiple:\n")
for iid, texto in items_3op.items():
    resp = cliente_glm.chat.completions.create(
        model="glm-5.2",
        messages=[
            {"role": "system", "content": sys_menu3},
            {"role": "user", "content": texto},
        ],
        max_tokens=800,
        extra_body={"reasoning_effort": "xhigh"}
    )
    out = resp.choices[0].message.content
    valor, estado = parsear_menu3(out)
    principio = clave[iid].get(valor, "?") if valor else "?"
    print(f"  {iid}: eligió {valor} = {principio} [{estado}]")

Prueba de humo GLM — 3 escenarios de opción múltiple:

  DIL3OP_01_bonos: eligió 1 = merito [ok]
  DIL3OP_03_fondos: eligió 2 = necesidad [ok]
  DIL3OP_07_sede: eligió 3 = igualdad [ok]


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

filas, hecho = [], 0
total = len(items_3op) * 10
print(f"Voy a hacer {total} llamadas (GLM, tanda 3 opciones).\n")

for iid, texto in items_3op.items():
    for rep in range(10):
        try:
            resp = cliente_glm.chat.completions.create(
                model="glm-5.2",
                messages=[
                    {"role": "system", "content": sys_menu3},
                    {"role": "user", "content": texto},
                ],
                max_tokens=800,
                extra_body={"reasoning_effort": "xhigh"}
            )
            out = resp.choices[0].message.content; err = None
        except Exception as e:
            out, err = None, str(e)
        valor, estado = parsear_menu3(out)
        principio = clave[iid].get(valor, None) if valor else None
        filas.append({
            "modelo": "glm-5.2", "modo": "xhigh", "item_id": iid, "formato": "menu3",
            "repeticion": rep, "valor": valor, "principio_elegido": principio,
            "estado_parseo": estado if err is None else "error",
            "error": err, "respuesta_cruda": out,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        hecho += 1
        time.sleep(0.3)

df = pd.DataFrame(filas)
df.to_csv("glm_ola2_3opciones.csv", index=False)
print(f"Listo. {len(df)} filas en glm_ola2_3opciones.csv")
print("\nElecciones por escenario:")
for iid in items_3op:
    sub = df[df.item_id==iid]
    from collections import Counter
    print(f"  {iid}: {dict(Counter(sub.principio_elegido.dropna()))}")

Voy a hacer 30 llamadas (GLM, tanda 3 opciones).

Listo. 30 filas en glm_ola2_3opciones.csv

Elecciones por escenario:
  DIL3OP_01_bonos: {'merito': 10}
  DIL3OP_03_fondos: {'necesidad': 10}
  DIL3OP_07_sede: {'igualdad': 10}
